In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Support running from this folder or the repository root.
OUTPUT_DIR = Path.cwd() if (Path.cwd() / "experiments.py").is_file() else Path.cwd() / "distilled"
RESULTS_PATH = OUTPUT_DIR / "results_cesnet.csv"
GEOMETRY_PATH = OUTPUT_DIR / "target_geometry_cesnet.csv"

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
})
colors = ["#D55E00", "#E69F00", "#0072B2", "#009E73"]
estimators = ["ERM", "DRO", "MRO", "SNR"]

In [ ]:
results_df = pd.read_csv(RESULTS_PATH).sort_values(["gain", "m"])
geometry_df = pd.read_csv(GEOMETRY_PATH).sort_values("q")
results_05 = results_df.loc[np.isclose(results_df["gain"], 0.5)].copy()
if results_05.empty:
    raise ValueError("Run experiments.py with gain 0.5 to create the main figure.")
assert not results_05.duplicated("m").any()
assert (results_05["m"] == 2 * results_05["q"]).all()
assert (results_05["p"] == results_05["q"] + 1).all()

results_05.loc[results_05["m"].eq(results_05["m"].max()),
               ["m", "q", "p", "n", "gain", "eps"] +
               [name.lower() + "_regret" for name in estimators]]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5, 3), constrained_layout=True)
for estimator, color in zip(estimators, colors):
    ax.plot(results_05["m"], results_05[estimator.lower() + "_regret"],
            color=color, label=estimator)
ax.set_xlabel(r"response dimension $m$")
ax.set_ylabel("target regret")
ax.set_title(r"CESNET: Regret vs. Response Dimension", pad=8)
ax.set_xticks([0, 64, 128, 192, 268])
ax.set_xlim(0, 275)
ax.set_ylim(bottom=0)
handles = [Line2D([], [], color=color, lw=2) for color in colors]
fig.legend(handles, estimators, loc="lower center", ncol=4,
           bbox_to_anchor=(0.5, -0.11), frameon=False,
           columnspacing=0.9, handlelength=1.7)
fig.savefig(OUTPUT_DIR / "fig3_plot.pdf", bbox_inches="tight")


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7.5, 3), constrained_layout=True,
                        gridspec_kw={"wspace": 0.07})
ax = axs[0]
ax.plot(geometry_df["m"], geometry_df["residual_effective_rank"], color=colors[2])
ax.set_xlabel(r"response dimension $m$")
ax.set_ylabel(r"$\operatorname{tr}(\Sigma_E)/\|\Sigma_E\|_{\mathrm{op}}$")
ax.set_title("Residual effective rank")
ax.set_xticks([0, 64, 128, 192, 268])
ax.set_xlim(0, 275)
ax.set_ylim(bottom=0)

ax = axs[1]
ax.plot(2*geometry_df["q"], geometry_df["kappa"], color=colors[0])
ax.set_xlabel(r"response dimension $m$")
ax.set_ylabel(r"$\kappa$")
ax.set_title("Target condition number")
ax.set_xticks([0, 64, 128, 192, 268])
ax.set_xlim(0, 275)
ax.set_ylim(bottom=0)
fig.savefig(OUTPUT_DIR / "fig4_plots.pdf", bbox_inches="tight")
